# Part 3: Spark Data Processing
### Banking Dataset - All 12 Questions
> PySpark runs inside Google Colab via the `pyspark` pip package.

In [ ]:
!pip install pyspark matplotlib seaborn pandas -q
print('PySpark installed ✓')

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StringType
import matplotlib.pyplot as plt, pandas as pd

spark = SparkSession.builder.appName('BankingSparkProcessing').master('local[*]').getOrCreate()
spark.sparkContext.setLogLevel('ERROR')
print('Spark version:', spark.version)

In [ ]:
from google.colab import files
uploaded = files.upload()  # upload bank.csv
df = spark.read.csv('bank.csv', header=True, inferSchema=True)
print(f'Rows: {df.count()}  Columns: {len(df.columns)}')

## Q1 – Data Loading and Basic Inspection

In [ ]:
df.show(5)
df.printSchema()
df.select('age','balance','duration','campaign','pdays','previous').describe().show()

## Q2 – Data Filtering and Column Operations

In [ ]:
# Filter balance > 1000
df_filtered = df.filter(F.col('balance') > 1000)
print(f'Clients with balance > 1000: {df_filtered.count()}')
df_filtered.show(5)

In [ ]:
# Quarter column
month_map = F.create_map(
    F.lit('jan'),F.lit('Q1'), F.lit('feb'),F.lit('Q1'), F.lit('mar'),F.lit('Q1'),
    F.lit('apr'),F.lit('Q2'), F.lit('may'),F.lit('Q2'), F.lit('jun'),F.lit('Q2'),
    F.lit('jul'),F.lit('Q3'), F.lit('aug'),F.lit('Q3'), F.lit('sep'),F.lit('Q3'),
    F.lit('oct'),F.lit('Q4'), F.lit('nov'),F.lit('Q4'), F.lit('dec'),F.lit('Q4')
)
df = df.withColumn('quarter', month_map[F.col('month')])
df.select('month','quarter').distinct().orderBy('month').show()

## Q3 – GroupBy and Aggregation

In [ ]:
# Avg balance and median age per job
df.groupBy('job').agg(
    F.round(F.avg('balance'),2).alias('avg_balance'),
    F.percentile_approx('age',0.5).alias('median_age'),
    F.count('*').alias('count')
).orderBy(F.desc('avg_balance')).show()

In [ ]:
# Subscribed clients per marital status
df.filter(F.col('y')=='yes').groupBy('marital').count().orderBy(F.desc('count')).show()

## Q4 – UDF to Categorize Age Groups

In [ ]:
@F.udf(StringType())
def age_group(age):
    if age is None: return 'unknown'
    return '<30' if age < 30 else ('30-60' if age <= 60 else '>60')

df = df.withColumn('age_group', age_group(F.col('age')))
df.groupBy('age_group').count().orderBy('age_group').show()
print('age_group column added with categories: <30, 30-60, >60')

## Q5 – Advanced Data Transformations

In [ ]:
# Subscription rate by education
df.groupBy('education').agg(
    F.count('*').alias('total'),
    F.sum(F.when(F.col('y')=='yes',1).otherwise(0)).alias('subscribed')
).withColumn('sub_rate_%', F.round(F.col('subscribed')*100.0/F.col('total'),2)
).orderBy(F.desc('sub_rate_%')).show()

In [ ]:
# Top 3 professions by loan default rate
df.groupBy('job').agg(
    F.count('*').alias('total'),
    F.sum(F.when(F.col('default')=='yes',1).otherwise(0)).alias('defaults')
).withColumn('default_rate_%',F.round(F.col('defaults')*100.0/F.col('total'),2)
).orderBy(F.desc('default_rate_%')).limit(3).show()

## Q6 – String Manipulation

In [ ]:
df = df.withColumn('job_marital', F.concat_ws('_', 'job','marital'))
df = df.withColumn('contact_upper', F.upper('contact'))
df.select('job','marital','job_marital','contact','contact_upper').show(5)

## Q7 – Data Visualization

In [ ]:
# Convert to Pandas and plot
job_counts = df.groupBy('job').count().orderBy(F.desc('count')).toPandas()
plt.figure(figsize=(13,5))
plt.bar(job_counts['job'], job_counts['count'], color='steelblue', edgecolor='black')
plt.xticks(rotation=45, ha='right')
plt.title('Count of Clients by Job Type', fontsize=14)
plt.xlabel('Job Type'); plt.ylabel('Count')
plt.tight_layout(); plt.savefig('spark_clients_by_job.png',dpi=150); plt.show()

## Q8 – Complex Queries for Insights

In [ ]:
# Month with highest contacts + success rate
df.groupBy('month').agg(
    F.count('*').alias('contacts'),
    F.round(F.sum(F.when(F.col('y')=='yes',1).otherwise(0))*100.0/F.count('*'),2).alias('success_%')
).orderBy(F.desc('contacts')).show(5)

# Avg duration: subscribed vs not
df.groupBy('y').agg(F.round(F.avg('duration'),2).alias('avg_duration_sec')).show()

## Q9 – Correlation Between Age and Balance

In [ ]:
corr = df.stat.corr('age','balance')
print(f'Pearson correlation (age vs balance): {corr:.4f}')
print('A value near 0 indicates no strong linear relationship.')

## Q10 – Exploring Loan Defaults

In [ ]:
default_counts = df.groupBy('default').count().toPandas()
print(default_counts)
plt.figure(figsize=(6,4))
plt.bar(default_counts['default'], default_counts['count'], color=['green','red'], edgecolor='black')
plt.title('Credit Default Distribution'); plt.xlabel('Default'); plt.ylabel('Count')
plt.tight_layout(); plt.savefig('spark_default_dist.png',dpi=150); plt.show()

## Q11 – Contact Method Analysis

In [ ]:
df.groupBy('contact').agg(
    F.count('*').alias('total'),
    F.sum(F.when(F.col('y')=='yes',1).otherwise(0)).alias('subscribed'),
    F.round(F.sum(F.when(F.col('y')=='yes',1.0).otherwise(0))*100/F.count('*'),2).alias('success_%')
).orderBy(F.desc('success_%')).show()

## Q12 – Spark SQL with Temporary View

In [ ]:
df.createOrReplaceTempView('bank_view')
# Avg balance by age group
spark.sql("""
    SELECT age_group,
           ROUND(AVG(balance),2) AS avg_balance,
           COUNT(*) AS client_count
    FROM bank_view
    GROUP BY age_group
    ORDER BY age_group
""").show()

# Most common job types
spark.sql("""
    SELECT job, COUNT(*) AS count
    FROM bank_view
    GROUP BY job
    ORDER BY count DESC
    LIMIT 5
""").show()

In [ ]:
spark.stop()
print('All 12 Spark Processing questions complete ✓')